# Silver Layer — Hospital Data Inspector

Queries `s3://silver/hospitals.parquet` directly from MinIO using DuckDB.

In [1]:
import os

import duckdb

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

path = "s3://silver/hospitals.parquet"

con = duckdb.connect()
for ext in ("spatial", "httpfs"):
    con.install_extension(ext)
    con.load_extension(ext)

endpoint = os.environ.get("MINIO_ENDPOINT", "localhost:9000")
con.sql(f"""
    CREATE SECRET minio (
        TYPE s3,
        KEY_ID '{os.environ["MINIO_ACCESSKEY"]}',
        SECRET '{os.environ["MINIO_SECRETKEY"]}',
        ENDPOINT '{endpoint}',
        URL_STYLE 'path',
        USE_SSL false
    )
""")

%sql con --alias duckdb

## Schema

In [2]:
%%sql
SELECT * FROM (DESCRIBE SELECT * FROM '{{path}}')

,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,hospital_name,VARCHAR,YES,None,None,None
2,geom,GEOMETRY('OGC:CRS84'),YES,None,None,None
3,geom_3035,GEOMETRY('EPSG:3035'),YES,None,None,None
4,country,VARCHAR,YES,None,None,None
5,city,VARCHAR,YES,None,None,None
6,street,VARCHAR,YES,None,None,None
7,postcode,VARCHAR,YES,None,None,None
8,house_number,VARCHAR,YES,None,None,None
9,healthcare_specialty,VARCHAR,YES,None,None,None


## Row Count

In [3]:
%%sql
SELECT COUNT(*) AS total FROM '{{path}}'

,total
0,1823


## Row Count per Country

In [4]:
%%sql
SELECT country, COUNT(*) AS total
FROM '{{path}}'
GROUP BY country
ORDER BY total DESC

,country,total
0,England,1474
1,Scotland,193
2,Wales,116
3,Northern Ireland,40


## Sample Rows

In [5]:
%%sql
SELECT
    hospital_name,
    ST_AsText(geom) AS geom_wkt,
    ST_AsText(geom_3035) AS geom_3035_wkt,
    country,
    city,
    bronze_path
FROM '{{path}}'
LIMIT 5

,hospital_name,geom_wkt,geom_3035_wkt,country,city,bronze_path
0,"""Mid-Ulster Hospital""",POINT (-6.6141307 54.761817),POINT (3262109.3604772673 3640010.0182276745),Northern Ireland,Unknown,s3://bronze/northern_ireland/2026-06-07T09-37-...
1,"""Braid Valley Care Complex""",POINT (-6.272589 54.8749648),POINT (3286318.977138628 3647343.518214845),Northern Ireland,"""Ballymena""",s3://bronze/northern_ireland/2026-06-07T09-37-...
2,"""Mountfern Complex""",POINT (-6.6575017 55.1207975),POINT (3268760.885674407 3679658.781251918),Northern Ireland,Unknown,s3://bronze/northern_ireland/2026-06-07T09-37-...
3,"""Robinson Hospital""",POINT (-6.5058616 55.072073),POINT (3276881.7277561235 3672151.3916508616),Northern Ireland,Unknown,s3://bronze/northern_ireland/2026-06-07T09-37-...
4,"""Lakeview""",POINT (-7.2826835 55.0153949),POINT (3227288.0733326357 3677551.8558389614),Northern Ireland,Unknown,s3://bronze/northern_ireland/2026-06-07T09-37-...


## Null Counts

In [6]:
%%sql
SELECT
    COUNT(*) FILTER (WHERE id IS NULL)                   AS id_nulls,
    COUNT(*) FILTER (WHERE hospital_name IS NULL)        AS hospital_name_nulls,
    COUNT(*) FILTER (WHERE geom IS NULL)                 AS geom_nulls,
    COUNT(*) FILTER (WHERE geom_3035 IS NULL)            AS geom_3035_nulls,
    COUNT(*) FILTER (WHERE city IS NULL)                 AS city_nulls,
    COUNT(*) FILTER (WHERE postcode IS NULL)             AS postcode_nulls
FROM '{{path}}'

,id_nulls,hospital_name_nulls,geom_nulls,geom_3035_nulls,city_nulls,postcode_nulls
0,0,0,0,0,0,0


## "Unknown" Counts

In [7]:
%%sql
SELECT
    COUNT(*) FILTER (WHERE hospital_name = 'Unknown')          AS hospital_name_unknowns,
    COUNT(*) FILTER (WHERE city = 'Unknown')                   AS city_unknowns,
    COUNT(*) FILTER (WHERE postcode = 'Unknown')               AS postcode_unknowns,
    COUNT(*) FILTER (WHERE healthcare_specialty = 'Unknown')   AS healthcare_specialty_unknowns
FROM '{{path}}'

,hospital_name_unknowns,city_unknowns,postcode_unknowns,healthcare_specialty_unknowns
0,80,878,812,1644


## Hospitals Within 50km of Lancaster

Uses the `geom_3035` column (EPSG:3035, metric) so that `ST_Distance` returns metres.

In [8]:
%%sql
WITH lancaster AS (
    SELECT ST_Transform(ST_Point(-2.8013499, 54.0488219), 'EPSG:4326', 'EPSG:3035', true) AS geom
)
SELECT
    h.hospital_name,
    h.city,
    h.postcode,
    ROUND(ST_Distance(h.geom_3035, lancaster.geom) / 1000.0, 1) AS distance_km,
    ST_AsText(h.geom) AS geom_wkt
FROM '{{path}}' AS h, lancaster
WHERE ST_Distance(h.geom_3035, lancaster.geom) <= 50000
ORDER BY distance_km

,hospital_name,city,postcode,distance_km,geom_wkt
0,"""The Lancaster Hospital""","""Lancaster""","""LA1 3RH""",0.7,POINT (-2.7952469 54.0431418)
1,"""Royal Lancaster Infirmary""","""Lancaster""","""LA1 4RP""",0.7,POINT (-2.8005027 54.0426284)
2,"""Ashton Community Care Centre""","""Lancaster""","""LA1 4JT""",1.0,POINT (-2.7987449 54.0396994)
3,"""Dacrelands Clinic""","""Lancaster""","""LA1 2DU""",1.2,POINT (-2.8006616 54.0598681)
4,"""The Orchard""","""Lancaster""","""LA1 4JJ""",1.9,POINT (-2.8040055 54.0321496)
5,"""DeVitre House""","""Lancaster""","""LA1 5AL""",1.9,POINT (-2.8026259 54.0315277)
6,"""Queen Victoria Hospital""","""Morecambe""","""LA4 5NN""",4.6,POINT (-2.8588556 54.0727631)
7,"""The Cove""","""Heysham""","""LA3 2SL""",6.1,POINT (-2.8922193 54.0351442)
8,"""Fleetwood Hospital""","""Fleetwood""","""FY7 6BE""",19.3,POINT (-3.0090102 53.9267965)
9,Unknown,Unknown,Unknown,25.6,POINT (-3.0980734 54.1987391)
